# Domain Probe Experiments

Interactive notebook for running the frozen-feature sanity checks without rebuilding long CLI commands each time.

The notebook uses the same code as `diagnostics/domain_sanity_check.py`, so results should stay aligned with the script.

In [13]:
from pathlib import Path
import importlib
import json
import sys

import pandas as pd

repo_root = Path.cwd()
if repo_root.name == "diagnostics":
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from diagnostics import domain_sanity_check as dsc

importlib.reload(dsc);

In [35]:
args = dsc.parse_args([])
time = pd.Timestamp.now().strftime("%Y-%m-%d_%H-%M-%S")

# Backbone / dataset
args.model_type = "vanilla"
args.model_path = None
args.seed = 42
args.batch_size = 128
args.max_samples_per_split = None

# Probe selection
args.probes = ["torch-mlp"]

# Torch probe training setup
args.torch_probe_epochs = 50
args.torch_probe_learning_rate = 1e-4
args.torch_probe_optimizer = "adamw"
args.torch_probe_weight_decay = None  # disable weight decay
args.torch_probe_momentum = 0.9
args.torch_probe_class_weighting = "balanced"

# Probe architecture
args.torch_probe_hidden_dims = [32]
args.torch_probe_activation = "relu"
args.torch_probe_leaky_relu_slope = None
args.torch_probe_layer_norm = False
args.torch_probe_standardize = False

# Logging / visualization
args.torch_probe_show_steps = True
args.torch_probe_log_interval = 10
args.torch_probe_save_history = True
args.torch_probe_umap_dir = Path(f"diagnostics/torch_mlp_{time}_umap")
args.torch_probe_umap_split = "val"
args.torch_probe_umap_interval = 10
args.torch_probe_umap_representative_points = 8

# Optional report path
args.output_json = Path(f"torch_mlp_{time}_umap/report.json")

print("Running with the following arguments:")
print(vars(args))

Running with the following arguments:
{'model_type': 'vanilla', 'model_path': None, 'simulation_dataset_dir': PosixPath('dataset/simulation'), 'experiment_dataset_dir': PosixPath('dataset/experiment'), 'seed': 42, 'batch_size': 128, 'device': 'cuda', 'max_samples_per_split': None, 'torch_probe_epochs': 50, 'torch_probe_learning_rate': 0.0001, 'torch_probe_hidden_dim': 64, 'torch_probe_hidden_dims': [32], 'torch_probe_activation': 'relu', 'torch_probe_leaky_relu_slope': None, 'torch_probe_layer_norm': False, 'torch_probe_standardize': False, 'torch_probe_class_weighting': 'balanced', 'torch_probe_optimizer': 'adamw', 'torch_probe_weight_decay': None, 'torch_probe_momentum': 0.9, 'torch_probe_show_steps': True, 'torch_probe_log_interval': 10, 'torch_probe_save_history': True, 'torch_probe_umap_dir': PosixPath('diagnostics/torch_mlp_2026-04-13_00-19-38_umap'), 'torch_probe_umap_split': 'val', 'torch_probe_umap_interval': 10, 'torch_probe_umap_representative_points': 8, 'torch_probe_umap_g

In [36]:
report = dsc.run_probe_analysis(args)

if args.output_json is not None:
    dsc.save_report(report, args.output_json)

dsc.print_report_summary(report)

Loading the model: /data1/bowenyu/XPCS_ODE/models/Vanilla_XPCS_best_20260408-164950.pt


/home/bowenyu/miniconda3/envs/mace_env_310/lib/python3.10/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


[torch_mlp epoch 1/50] train_loss=0.694151 train_bal=0.500 train_pred_exp=0.000 val_loss=0.696403 val_bal=0.500 val_pred_exp=0.000 val_p(exp|sim)=0.473 val_p(exp|exp)=0.474
[torch_mlp epoch 11/50] train_loss=0.693217 train_bal=0.500 train_pred_exp=0.000 val_loss=0.693715 val_bal=0.500 val_pred_exp=0.000 val_p(exp|sim)=0.494 val_p(exp|exp)=0.494
[torch_mlp epoch 21/50] train_loss=0.693116 train_bal=0.500 train_pred_exp=0.000 val_loss=0.693410 val_bal=0.500 val_pred_exp=0.000 val_p(exp|sim)=0.497 val_p(exp|exp)=0.497
[torch_mlp epoch 31/50] train_loss=0.693056 train_bal=0.500 train_pred_exp=0.000 val_loss=0.694074 val_bal=0.500 val_pred_exp=0.000 val_p(exp|sim)=0.488 val_p(exp|exp)=0.488
[torch_mlp epoch 41/50] train_loss=0.692891 train_bal=0.500 train_pred_exp=0.000 val_loss=0.693124 val_bal=0.500 val_pred_exp=0.000 val_p(exp|sim)=0.497 val_p(exp|exp)=0.497
[torch_mlp epoch 50/50] train_loss=0.692906 train_bal=0.500 train_pred_exp=0.000 val_loss=0.694153 val_bal=0.500 val_pred_exp=0.000

In [ ]:
probe_key = "torch_mlp_probe"
print(report[probe_key]["probe_model_info"]["architecture"])
print(json.dumps(report[probe_key]["probe_config"], indent=2))

In [ ]:
history = pd.DataFrame(report[probe_key].get("history", []))
history.tail()